## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
fatal: unable to access 'https://github.com/Lv1g1/RecSys-Challenge-2025.git/': Could not resolve host: github.com


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


## **Load Data**

In [5]:
URM_train, URM_val = paths.load_holdout_split()

## **Load Models**

In [ ]:
from Recommenders.NonPersonalizedRecommender import TopPop
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender

from Recommenders.SLIM.Cython.SLIM_BPR_Cython import SLIM_BPR_Cython
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
from Recommenders.MatrixFactorization.PureSVDRecommender import PureSVDRecommender
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender
from Recommenders.Neural.MultVAERecommender import MultVAERecommender

models = {}

TRAIN = False
model_folder = os.path.join(paths.MODEL_DIR, "xg_boost_all_data")

Tensorflow is not available


In [7]:
import json

def get_best_params(json_path: str) -> dict:
    with open(json_path, "r") as f:
        data = json.load(f)
        
        best_study = None
        for _, values in data.items():
            if best_study is None:
                best_study = values
                continue

            if values["best_score"] > best_study["best_score"]:
                best_study = values
                
    return best_study["best_params"]

In [8]:
if TRAIN:
    # TopPop
    model = TopPop(URM_train+URM_val)
    model.fit()
    models["TopPop"] = model

    # KNN
    KNN_similarities = ["cosine", "jaccard", "asymmetric", "tversky", "dice"]
    for sim in KNN_similarities:
        path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/UserKNN_{sim}.json")
        best_params = get_best_params(path)
                
        user_model = UserKNNCFRecommender(URM_train+URM_val)
        user_model.fit(**best_params)
        models[f"UserKNNCF_{sim}"] = user_model
        
        path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/ItemKNN_{sim}.json")
        best_params = get_best_params(path)

        item_model = ItemKNNCFRecommender(URM_train+URM_val)
        item_model.fit(**best_params)
        models[f"ItemKNNCF_{sim}"] = item_model

    # P3alpha
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/P3alpha.json")
    best_params = get_best_params(path)
    p3alpha_model = P3alphaRecommender(URM_train+URM_val)
    p3alpha_model.fit(**best_params)
    models["P3alpha"] = p3alpha_model

    # RP3beta
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/RP3beta.json")
    best_params = get_best_params(path)
    rp3beta_model = RP3betaRecommender(URM_train+URM_val)
    rp3beta_model.fit(**best_params)
    models["RP3beta"] = rp3beta_model

    # EASE-R
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/EASE_R.json")
    best_params = get_best_params(path)
    ease_r_model = EASE_R_Recommender(URM_train+URM_val)
    ease_r_model.fit(**best_params)
    models["EASE_R"] = ease_r_model

    # SLIM ElasticNet
    path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/SLIMElasticNet.json")
    best_params = get_best_params(path)
    slim_en_model = MultiThreadSLIM_SLIMElasticNetRecommender(URM_train+URM_val)
    slim_en_model.fit(**best_params, workers=8)
    models["SLIMElasticNet"] = slim_en_model

    # Save models
    for model_name, model in models.items():
        model.save_model(model_folder, model_name)

else:
    models["TopPop"] = TopPop(URM_train+URM_val)
    models["TopPop"].load_model(model_folder, "TopPop")
    for sim in ["cosine", "jaccard", "asymmetric", "tversky", "dice"]:
        models[f"UserKNNCF_{sim}"] = UserKNNCFRecommender(URM_train+URM_val)
        models[f"UserKNNCF_{sim}"].load_model(model_folder, f"UserKNNCF_{sim}")
        models[f"ItemKNNCF_{sim}"] = ItemKNNCFRecommender(URM_train+URM_val)
        models[f"ItemKNNCF_{sim}"].load_model(model_folder, f"ItemKNNCF_{sim}")
    models["RP3beta"] = RP3betaRecommender(URM_train+URM_val)
    models["RP3beta"].load_model(model_folder, "RP3beta")
    models["EASE_R"] = EASE_R_Recommender(URM_train+URM_val)
    models["EASE_R"].load_model(model_folder, "EASE_R")
    models["SLIMElasticNet"] = MultiThreadSLIM_SLIMElasticNetRecommender(URM_train+URM_val)
    models["SLIMElasticNet"].load_model(model_folder, "SLIMElasticNet")

Similarity column 27095 (100.0%), 1458.00 column/sec. Elapsed time 18.58 sec
Similarity column 6969 (100.0%), 3570.44 column/sec. Elapsed time 1.95 sec
Similarity column 27095 (100.0%), 1456.16 column/sec. Elapsed time 18.61 sec
Similarity column 6969 (100.0%), 3475.55 column/sec. Elapsed time 2.01 sec
Similarity column 27095 (100.0%), 1477.46 column/sec. Elapsed time 18.34 sec
Similarity column 6969 (100.0%), 3682.71 column/sec. Elapsed time 1.89 sec
Similarity column 27095 (100.0%), 1492.79 column/sec. Elapsed time 18.15 sec
Similarity column 6969 (100.0%), 3432.38 column/sec. Elapsed time 2.03 sec
Similarity column 27095 (100.0%), 1395.89 column/sec. Elapsed time 19.41 sec
Similarity column 6969 (100.0%), 3473.32 column/sec. Elapsed time 2.01 sec
P3alphaRecommender: Similarity column 6969 (100.0%), 2791.63 column/sec. Elapsed time 2.50 sec
RP3betaRecommender: Similarity column 6969 (100.0%), 3190.31 column/sec. Elapsed time 2.18 sec
EASE_R_Recommender: Fitting model... 
EASE_R_Recom

100%|█████████▉| 6968/6969 [02:14<00:00, 51.95it/s] 


TopPopRecommender: Saving model in file '/home/luigi/RecSys/models/xg_boost_all_dataTopPop'
TopPopRecommender: Saving complete
UserKNNCFRecommender: Saving model in file '/home/luigi/RecSys/models/xg_boost_all_dataUserKNNCF_cosine'
UserKNNCFRecommender: Saving complete
ItemKNNCFRecommender: Saving model in file '/home/luigi/RecSys/models/xg_boost_all_dataItemKNNCF_cosine'
ItemKNNCFRecommender: Saving complete
UserKNNCFRecommender: Saving model in file '/home/luigi/RecSys/models/xg_boost_all_dataUserKNNCF_jaccard'
UserKNNCFRecommender: Saving complete
ItemKNNCFRecommender: Saving model in file '/home/luigi/RecSys/models/xg_boost_all_dataItemKNNCF_jaccard'
ItemKNNCFRecommender: Saving complete
UserKNNCFRecommender: Saving model in file '/home/luigi/RecSys/models/xg_boost_all_dataUserKNNCF_asymmetric'
UserKNNCFRecommender: Saving complete
ItemKNNCFRecommender: Saving model in file '/home/luigi/RecSys/models/xg_boost_all_dataItemKNNCF_asymmetric'
ItemKNNCFRecommender: Saving complete
UserK

In [9]:
print(f"{len(models)} models")
models.keys()

15 models


dict_keys(['TopPop', 'UserKNNCF_cosine', 'ItemKNNCF_cosine', 'UserKNNCF_jaccard', 'ItemKNNCF_jaccard', 'UserKNNCF_asymmetric', 'ItemKNNCF_asymmetric', 'UserKNNCF_tversky', 'ItemKNNCF_tversky', 'UserKNNCF_dice', 'ItemKNNCF_dice', 'P3alpha', 'RP3beta', 'EASE_R', 'SLIMElasticNet'])

## **Selection phase**

In [6]:
import numpy as np
import scipy.sparse as sps
import pandas as pd
import tqdm
from scipy import stats

selection_models = ['ItemKNNCF_asymmetric', 'UserKNNCF_tversky', 'SLIMElasticNet', 'TopPop']

In [11]:
cutoff = 100
n_users, n_items = (URM_train+URM_val).shape

pred_dataframe = pd.DataFrame(index=range(0, n_users), columns = ["ItemID"])
pred_dataframe.index.name='UserID'

for user_id in tqdm.notebook.tqdm(range(n_users)):    
    recommendations = None
    for model_name in selection_models:
        recommender = models[model_name]
        recommendations_model = recommender.recommend(user_id, cutoff = cutoff)

        if recommendations is None:
            recommendations = recommendations_model
        else:
            recommendations = np.union1d(recommendations, recommendations_model)

    if user_id == 0:
        print(recommendations)
    
    pred_dataframe.loc[user_id, "ItemID"] = recommendations
    
pred_dataframe = pred_dataframe.explode("ItemID")
pred_dataframe

  0%|          | 0/27095 [00:00<?, ?it/s]

[  13   76   86   89   91  200  262  275  282  283  291  333  344  355
  372  457  479  500  531  536  594  596  608  625  647  665  729  773
  775  814  828  868  886  891  909  955 1009 1013 1038 1043 1106 1131
 1146 1183 1252 1257 1294 1328 1418 1445 1458 1489 1501 1503 1528 1539
 1554 1560 1568 1585 1602 1608 1632 1656 1700 1703 1739 1765 1779 1930
 1940 1958 1983 2003 2026 2052 2061 2080 2101 2110 2150 2162 2168 2223
 2228 2232 2233 2252 2258 2329 2369 2385 2392 2399 2432 2444 2499 2530
 2539 2555 2574 2587 2628 2653 2699 2709 2779 2824 2835 2842 2917 3019
 3049 3218 3236 3239 3256 3300 3316 3373 3385 3410 3436 3441 3496 3512
 3514 3526 3551 3590 3602 3604 3607 3636 3656 3666 3795 3824 3942 3966
 4023 4045 4170 4180 4200 4220 4270 4272 4293 4312 4316 4342 4349 4351
 4373 4381 4486 4533 4534 4545 4570 4709 4720 4772 4796 4813 4835 4877
 4901 4912 4925 4996 5010 5022 5027 5064 5065 5069 5076 5108 5122 5166
 5167 5252 5258 5295 5323 5326 5329 5340 5342 5369 5407 5460 5466 5469
 5474 

,ItemID
UserID,
0,13
0,76
0,86
0,89
0,91
...,...
27094,6675
27094,6724
27094,6771


In [16]:
pred_dataframe = pred_dataframe.reset_index()

In [17]:
import gc

def get_user_batches(user_ids, batch_size=1000):
    for i in range(0, len(user_ids), batch_size):
        yield user_ids[i:i + batch_size]

# Extract unique (UserID, ItemID) candidates
feature_candidates = pred_dataframe[['UserID', 'ItemID']].copy()
N_CANDIDATES = len(feature_candidates)
new_features_to_merge = [] 
unique_users = feature_candidates['UserID'].unique()
BATCH_SIZE = 1000 # Define your batch size here

# --- 2. Refactored Feature Extraction Loop (Model by Model) ---
for label, recommender in models.items():
    print(f"Processing features for model: {label}")
    
    # Pre-allocate arrays for the new features (only size of the candidate set)
    current_scores = np.zeros(N_CANDIDATES, dtype=np.float32)
    current_ranks = np.zeros(N_CANDIDATES, dtype=np.int32)
    
    # Iterate over user batches
    for user_batch in tqdm.tqdm(list(get_user_batches(unique_users, BATCH_SIZE)), desc=f"Processing Batches for {label}", leave=False):
        
        scores_batch = recommender._compute_item_score(user_id_array=user_batch)

        # Normalize
        norm_factor = np.linalg.norm(scores_batch, np.inf, axis=1, keepdims=True) + 1e-6
        linf_scores_batch = scores_batch / norm_factor

        # Remove 
        for i, user_id in enumerate(user_batch):
            linf_scores_batch[i, :] = recommender._remove_seen_on_scores(user_id, linf_scores_batch[i, :])

        # Calculate rank
        rank_order = np.argsort(linf_scores_batch, axis=1)[:, ::-1]
        rank_matrix_batch = np.zeros_like(linf_scores_batch, dtype=np.int32)

        for i in range(len(user_batch)):
            rank_matrix_batch[i, rank_order[i, :]] = np.arange(len(scores_batch[i]))

        # Create a mapping from UserID to its index within the current batch (0 to N_batch-1)
        user_to_batch_index = {uid: i for i, uid in enumerate(user_batch)}
        
        # Get all candidates rows belonging to the current batch of users
        batch_mask = feature_candidates['UserID'].isin(user_batch)
        batch_candidate_indices = feature_candidates.index[batch_mask].to_numpy()
        
        # Identify ItemIDs and map UserIDs to the batch index
        candidate_item_ids = feature_candidates.loc[batch_candidate_indices, 'ItemID'].to_numpy().astype(np.int32)
        candidate_user_ids = feature_candidates.loc[batch_candidate_indices, 'UserID'].to_numpy()
        
        # Map the UserIDs to their row index in the N_batch dimension (0, 1, 2, ...)
        batch_row_indices = np.array([user_to_batch_index[uid] for uid in candidate_user_ids])
        
        # Perform the final indexed lookup (2D indexing on the batch matrices)
        current_scores[batch_candidate_indices] = linf_scores_batch[batch_row_indices, candidate_item_ids]
        current_ranks[batch_candidate_indices] = rank_matrix_batch[batch_row_indices, candidate_item_ids]
        
        # CRITICAL: Delete the temporary large matrices
        del scores_batch, linf_scores_batch, rank_matrix_batch, rank_order
        gc.collect()

    # Finalize and store the new features
    df_features = pd.DataFrame({
        f"{label}_Score": current_scores,
        f"{label}_RankPosition": current_ranks,
        f"{label}_Recommended": (current_ranks < 10).astype(int)
    })
    new_features_to_merge.append(df_features)
    
    # Cleanup for the next model
    del current_scores, current_ranks, df_features
    gc.collect()

# Concatenate all new feature DataFrames along the columns axis
all_new_features = pd.concat(new_features_to_merge, axis=1)

# Add the new feature columns back to the main DataFrame
for col in all_new_features.columns:
    pred_dataframe[col] = all_new_features[col].values
    
pred_dataframe = pred_dataframe.set_index('UserID') 

# Final cleanup
del all_new_features, feature_candidates
gc.collect()

Processing features for model: TopPop


Processing features for model: UserKNNCF_cosine


Processing features for model: ItemKNNCF_cosine


Processing features for model: UserKNNCF_jaccard


Processing features for model: ItemKNNCF_jaccard


...

Processing features for model: P3alpha


Processing features for model: RP3beta


Processing features for model: EASE_R


Processing features for model: SLIMElasticNet


0

In [18]:
def calculate_item_item_features_batched(training_dataframe, URM_train, models, batch_size=1000):
    stats_list = ['Avg', 'Max', 'Min', 'Std', 'Skew', 'Kurtosis']
    all_user_ids = training_dataframe.index.unique().to_numpy()
    
    # Store initial features to merge at the end
    df_features_to_merge = training_dataframe.reset_index()[['UserID', 'ItemID']].copy()
    N_CANDIDATES = len(df_features_to_merge)

    for similarity_type, recommender in [
            ('ItemKNN_cosine', models['ItemKNNCF_cosine']), 
            ('ItemKNN_tversky', models['ItemKNNCF_tversky']),
            ('RP3beta', models['RP3beta'])
        ]:
        print(f"Adding {similarity_type} similarity features (Batched & Sparse Optimized)...")
        
        # Use the SPARSE Similarity Matrix
        W_sparse = recommender.W_sparse.tocsr() 
        
        # Prepare storage for the new feature columns (size: N_candidates)
        new_features = {
            f"{stat}SimilarityToSeen{similarity_type}": np.zeros(N_CANDIDATES, dtype=np.float32) 
            for stat in stats_list
        }
        
        # Loop over user batches
        user_batches = list(get_user_batches(all_user_ids, batch_size))
        for user_batch in tqdm.tqdm(user_batches, desc=f"Processing Batches for {similarity_type}"):
            # Identify all candidates belonging to the current batch of users
            batch_mask = df_features_to_merge['UserID'].isin(user_batch)
            
            # Initialize storage for the stats arrays for this batch
            batch_stats_results = {stat: [] for stat in stats_list}
            
            for user_id in user_batch:
                # Get the candidate indices within the *full* df_features_to_merge for this *single* user
                user_candidate_indices = df_features_to_merge.index[
                    (df_features_to_merge['UserID'] == user_id) & batch_mask
                ].to_numpy()
                
                if len(user_candidate_indices) == 0:
                    continue # No candidates for this user in the training set
                
                # Items the user has interacted with
                seen_items = URM_train.getrow(user_id).nonzero()[1]
                
                # Get candidate item IDs for this user only
                current_candidate_item_ids = df_features_to_merge.loc[user_candidate_indices, 'ItemID'].values.astype(np.int32)
                
                if len(seen_items) == 0:
                    # Append zeros for all candidates of this user
                    num_candidates = len(user_candidate_indices)
                    for stat in stats_list:
                         batch_stats_results[stat].append(np.zeros(num_candidates, dtype=np.float32))
                    continue
                    
                # Extract Similarity Scores using SPARSE Slicing (Vectorized across N_candidates)
                similarities_candidate_rows = W_sparse[current_candidate_item_ids, :] 
                similarities_slice = similarities_candidate_rows[:, seen_items]
                similarities = similarities_slice.toarray()
                
                # Calculate statistics
                results = {
                    "Avg": similarities.mean(axis=1),
                    "Max": similarities.max(axis=1),
                    "Min": similarities.min(axis=1),
                    "Std": similarities.std(axis=1),
                    "Skew": stats.skew(similarities, axis=1),
                    "Kurtosis": stats.kurtosis(similarities, axis=1)
                }
                
                # Store results for later assignment
                for stat, value in results.items():
                    batch_stats_results[stat].append(value)

                del similarities, similarities_slice, similarities_candidate_rows
                gc.collect()

            #  Concatenate and assign the results for the whole batch
            if batch_stats_results['Avg']: # Check if any results were actually collected
                for stat in stats_list:
                    # Concatenate all user results for this stat into one array
                    stat_values = np.concatenate(batch_stats_results[stat])
                    
                    # Find the specific indices within the main feature array where these values go
                    batch_candidates_sorted = df_features_to_merge[batch_mask].index.to_numpy()
                    
                    # The gathered stat_values must correspond exactly to the order of batch_candidates_sorted
                    new_features[f"{stat}SimilarityToSeen{similarity_type}"][batch_candidates_sorted] = stat_values

        # Add the completed feature columns to the merge DataFrame
        for col_name, data in new_features.items():
            df_features_to_merge[col_name] = data

    # --- 5. Final Merge (Same as before) ---
    training_dataframe = pd.merge(training_dataframe.reset_index(), 
                                 df_features_to_merge, 
                                 on=['UserID', 'ItemID'], 
                                 how='left',
                                 suffixes=('_old', ''))

    training_dataframe = training_dataframe.set_index('UserID')
    
    del df_features_to_merge, W_sparse
    gc.collect()
    
    return training_dataframe

In [19]:
pred_dataframe = calculate_item_item_features_batched(pred_dataframe, URM_train+URM_val, models)

Adding ItemKNN_cosine similarity features (Batched & Sparse Optimized)...


Processing Batches for ItemKNN_cosine: 100%|██████████| 28/28 [31:44<00:00, 68.00s/it]


Adding ItemKNN_tversky similarity features (Batched & Sparse Optimized)...


Processing Batches for ItemKNN_tversky: 100%|██████████| 28/28 [31:37<00:00, 67.78s/it]


Adding RP3beta similarity features (Batched & Sparse Optimized)...


Processing Batches for RP3beta: 100%|██████████| 28/28 [31:36<00:00, 67.74s/it]


In [20]:
# Final Meta-Features

# A. Consensus Features (Fast)
recommended_columns = [col for col in pred_dataframe.columns if col.endswith('_Recommended')]
pred_dataframe['Counter_Recommended'] = pred_dataframe[recommended_columns].sum(axis=1).astype(int)

# B. Rank Position Statistics (Fast)
position_columns = [col for col in pred_dataframe.columns if col.endswith('_RankPosition')]
pred_dataframe['Mean_RankPosition'] = pred_dataframe[position_columns].mean(axis=1)
pred_dataframe['Std_RankPosition'] = pred_dataframe[position_columns].std(axis=1)
pred_dataframe['Skew_RankPosition'] = pred_dataframe[position_columns].skew(axis=1)
pred_dataframe['Kurtosis_RankPosition'] = pred_dataframe[position_columns].kurtosis(axis=1)
# C. Score Statistics (CRITICAL ADDITION)
score_columns = [col for col in pred_dataframe.columns if col.endswith('_Score')]

pred_dataframe['Mean_Score'] = pred_dataframe[score_columns].mean(axis=1)
pred_dataframe['Std_Score'] = pred_dataframe[score_columns].std(axis=1)
pred_dataframe['Skew_Score'] = pred_dataframe[score_columns].skew(axis=1)
pred_dataframe['Kurtosis_Score'] = pred_dataframe[score_columns].kurtosis(axis=1)


# D. Final Cleanup and Index Reset
# We ensure the index is a regular column and any intermediate index is removed.
# Assuming 'UserID' is the name of the index at this point from previous steps
if pred_dataframe.index.name == 'UserID':
    pred_dataframe = pred_dataframe.reset_index()
else:
    # If index is unnamed (from a previous reset), just reset it and ensure columns are unique
    pred_dataframe = pred_dataframe.reset_index(drop=True)

print("Feature Engineering Complete. 🎉")
print(f"Final DataFrame shape: {pred_dataframe.shape}")

Feature Engineering Complete. 🎉
Final DataFrame shape: (6308193, 74)


In [22]:
# Example check
assert 'UserID' in pred_dataframe.columns
assert 'ItemID' in pred_dataframe.columns
assert pred_dataframe.shape[0] == 6308193

print("\n## ⚠️ Missing Value Check (Should be near zero)")
print(pred_dataframe.isnull().sum().sort_values(ascending=False).head(10))

print("\n## 🎯 Rank and Consensus Checks")
# Max rank should be less than the total number of items (N_items)
print(f"Max Rank Position: {pred_dataframe['ItemKNNCF_cosine_RankPosition'].max()}") 
# Max consensus count should match the number of models used
print(f"Max Recommended Counter: {pred_dataframe['Counter_Recommended'].max()}")

print("\n## 📈 Similarity Feature Range Check")
# Example for Cosine Avg Similarity
print(f"Max Mean Cosine Sim: {pred_dataframe['AvgSimilarityToSeenItemKNN_cosine'].max()}")
print(f"Min Mean Cosine Sim: {pred_dataframe['AvgSimilarityToSeenItemKNN_cosine'].min()}")

print("\n## 📊 Meta-Feature Statistics (Descriptive)")
print(pred_dataframe[['Mean_RankPosition', 'Std_RankPosition', 'Skew_RankPosition']].describe())
print(pred_dataframe[['Mean_Score', 'Std_Score']].describe())


## ⚠️ Missing Value Check (Should be near zero)
SkewSimilarityToSeenItemKNN_tversky        3741326
KurtosisSimilarityToSeenItemKNN_tversky    3741326
SkewSimilarityToSeenRP3beta                2617110
KurtosisSimilarityToSeenRP3beta            2617110
KurtosisSimilarityToSeenItemKNN_cosine      210315
SkewSimilarityToSeenItemKNN_cosine          210315
TopPop_Score                                     0
TopPop_RankPosition                              0
UserID                                           0
ItemID                                           0
dtype: int64

## 🎯 Rank and Consensus Checks
Max Rank Position: 6950
Max Recommended Counter: 15

## 📈 Similarity Feature Range Check
Max Mean Cosine Sim: 0.4424787759780884
Min Mean Cosine Sim: 0.0

## 📊 Meta-Feature Statistics (Descriptive)
       Mean_RankPosition  Std_RankPosition  Skew_RankPosition
count       6.308193e+06      6.308193e+06       6.308193e+06
mean        1.037863e+03      1.183194e+03       1.616806e+00
std         

In [23]:
# Apply the fix immediately after the feature engineering steps
for col in pred_dataframe.columns:
    if 'SimilarityToSeen' in col and pred_dataframe[col].isnull().any():
        # Impute Skew/Kurtosis NaNs with 0.0
        pred_dataframe[col] = pred_dataframe[col].fillna(0.0)

print("NaNs in Skew/Kurtosis imputed to 0.0.")

NaNs in Skew/Kurtosis imputed to 0.0.


In [24]:
# Example check
assert 'UserID' in pred_dataframe.columns
assert 'ItemID' in pred_dataframe.columns
assert pred_dataframe.shape[0] == 6308193

print("\n## ⚠️ Missing Value Check (Should be near zero)")
print(pred_dataframe.isnull().sum().sort_values(ascending=False).head(10))

print("\n## 🎯 Rank and Consensus Checks")
# Max rank should be less than the total number of items (N_items)
print(f"Max Rank Position: {pred_dataframe['ItemKNNCF_cosine_RankPosition'].max()}") 
# Max consensus count should match the number of models used
print(f"Max Recommended Counter: {pred_dataframe['Counter_Recommended'].max()}")

print("\n## 📈 Similarity Feature Range Check")
# Example for Cosine Avg Similarity
print(f"Max Mean Cosine Sim: {pred_dataframe['AvgSimilarityToSeenItemKNN_cosine'].max()}")
print(f"Min Mean Cosine Sim: {pred_dataframe['AvgSimilarityToSeenItemKNN_cosine'].min()}")

print("\n## 📊 Meta-Feature Statistics (Descriptive)")
print(pred_dataframe[['Mean_RankPosition', 'Std_RankPosition', 'Skew_RankPosition']].describe())
print(pred_dataframe[['Mean_Score', 'Std_Score']].describe())


## ⚠️ Missing Value Check (Should be near zero)
UserID                           0
ItemID                           0
TopPop_Score                     0
TopPop_RankPosition              0
TopPop_Recommended               0
UserKNNCF_cosine_Score           0
UserKNNCF_cosine_RankPosition    0
UserKNNCF_cosine_Recommended     0
ItemKNNCF_cosine_Score           0
ItemKNNCF_cosine_RankPosition    0
dtype: int64

## 🎯 Rank and Consensus Checks
Max Rank Position: 6950
Max Recommended Counter: 15

## 📈 Similarity Feature Range Check
Max Mean Cosine Sim: 0.4424787759780884
Min Mean Cosine Sim: 0.0

## 📊 Meta-Feature Statistics (Descriptive)
       Mean_RankPosition  Std_RankPosition  Skew_RankPosition
count       6.308193e+06      6.308193e+06       6.308193e+06
mean        1.037863e+03      1.183194e+03       1.616806e+00
std         1.108935e+03      9.236697e+02       1.200470e+00
min         0.000000e+00      0.000000e+00      -3.831228e+00
25%         2.283333e+02      2.660687e+02      

In [25]:
pred_dataframe.head()

,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,UserKNNCF_cosine_Score,UserKNNCF_cosine_RankPosition,UserKNNCF_cosine_Recommended,ItemKNNCF_cosine_Score,ItemKNNCF_cosine_RankPosition,...,KurtosisSimilarityToSeenRP3beta,Counter_Recommended,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score
0,0,13,0.040529,1325,0,0.011817,2046,0,0.209142,432,...,68.941254,0,830.066667,761.575479,0.507599,-1.641766,0.076503,0.068276,0.634008,-1.030331
1,0,76,0.071208,820,0,0.139942,152,0,0.377958,94,...,91.010490,0,197.933333,187.226168,2.923757,9.774613,0.144521,0.087829,1.293600,2.546777
2,0,86,0.316490,75,0,0.213618,55,0,0.274281,237,...,91.010513,0,173.800000,141.200061,1.551597,1.966609,0.145828,0.086696,0.348141,-0.277668
3,0,89,0.207234,188,0,0.113880,216,0,0.435132,51,...,0.000000,0,173.666667,127.962420,2.384171,7.205749,0.148445,0.106124,1.286077,3.100661
4,0,91,0.281299,98,0,0.074746,395,0,0.412305,66,...,0.000000,0,242.200000,156.423509,0.666893,-0.490574,0.129555,0.115258,1.240246,1.230077


In [26]:
pred_dataframe.to_csv("XGboost_pred_dataframe.csv", index=False)

## **Make Predictions**

In [7]:
pred_dataframe = pd.read_csv("XGboost_pred_dataframe.csv")

In [8]:
pred_dataframe

,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,UserKNNCF_cosine_Score,UserKNNCF_cosine_RankPosition,UserKNNCF_cosine_Recommended,ItemKNNCF_cosine_Score,ItemKNNCF_cosine_RankPosition,...,KurtosisSimilarityToSeenRP3beta,Counter_Recommended,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score
0,0,13,0.040529,1325,0,0.011817,2046,0,0.209142,432,...,68.941250,0,830.066667,761.575479,0.507599,-1.641766,0.076503,0.068276,0.634008,-1.030331
1,0,76,0.071208,820,0,0.139942,152,0,0.377958,94,...,91.010490,0,197.933333,187.226168,2.923757,9.774613,0.144521,0.087829,1.293600,2.546776
2,0,86,0.316490,75,0,0.213618,55,0,0.274281,237,...,91.010510,0,173.800000,141.200061,1.551597,1.966609,0.145828,0.086696,0.348141,-0.277668
3,0,89,0.207234,188,0,0.113880,216,0,0.435132,51,...,0.000000,0,173.666667,127.962420,2.384171,7.205749,0.148445,0.106124,1.286077,3.100661
4,0,91,0.281299,98,0,0.074746,395,0,0.412305,66,...,0.000000,0,242.200000,156.423509,0.666893,-0.490574,0.129555,0.115258,1.240246,1.230077
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6308188,27094,6675,0.056019,949,0,0.256274,137,0,0.335531,212,...,162.091810,0,722.600000,1590.499284,3.674203,13.851994,0.178310,0.127575,0.062555,-1.500604
6308189,27094,6724,0.020829,1984,0,0.287453,109,0,0.435009,107,...,71.436540,0,319.666667,496.539693,3.088394,10.181651,0.218344,0.119803,-0.285048,-0.377831
6308190,27094,6771,0.390180,39,0,0.113013,572,0,0.097888,1311,...,309.003230,0,1037.200000,927.396371,1.692480,2.133404,0.097325,0.096581,2.063323,5.914947
6308191,27094,6920,0.030754,1536,0,0.127277,489,0,0.254413,387,...,115.298805,0,336.666667,365.466760,2.768140,9.121166,0.181998,0.082033,0.000524,-0.375361


In [9]:
import joblib

MODEL_DIR = "models"
MODEL_FILENAME = "xgb_meta_ranker_v1.joblib"
MODEL_PATH = os.path.join(MODEL_DIR, MODEL_FILENAME)

loaded_xgb_model = joblib.load(MODEL_PATH)

print(f"✅ XGBRanker model successfully loaded from disk.")

✅ XGBRanker model successfully loaded from disk.


In [10]:
class XGBoostRerankerRecommender:
    def __init__(self, URM_train, XGB_model, df, index_df):
        self.URM_train = URM_train
        self.df = df
        self.index_df = index_df
        self.XGB_model = XGB_model
        
    def recommend(self, user_ids, cutoff=10, remove_seen_flag=True, remove_top_pop_flag=True, remove_custom_items_flag=False):
        recommendations = []
        for user_id in user_ids:

            df_slice = self.df[self.index_df['UserID'] == user_id]
            items = self.index_df[self.index_df['UserID'] == user_id]['ItemID'].to_numpy()
            preds = self.XGB_model.predict(df_slice)
            recommendations.append(items[np.argsort(preds)[-cutoff:][::-1]].tolist())

        return np.array(recommendations)
    
    def get_URM_train(self):
        return self.URM_train

In [11]:
pred_dataframe = pred_dataframe.sort_values(by='UserID').reset_index(drop=True)

In [13]:
import gc
gc.collect()

744

In [11]:
index_df = pred_dataframe[['UserID','ItemID']].copy()

In [12]:
X_pred = pred_dataframe.drop(columns=["UserID", "ItemID", "P3alpha_Recommended", "P3alpha_RankPosition", "P3alpha_Score"], errors='ignore')

In [13]:
del pred_dataframe

import gc
gc.collect()

16

In [14]:
recommender = XGBoostRerankerRecommender(URM_train+URM_val, loaded_xgb_model, X_pred, index_df)

In [15]:
# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = recommender.recommend(user_ids=ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, 'xg_boost_submission_trial_1.csv'), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")